In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("joins").getOrCreate()

1. Calculate Bonus and Total Salary

Input: A DataFrame with columns: `emp_id`, `name`, `salary`

Task: Add a new column `bonus` (10% of salary) and calculate `total_salary = salary + bonus`.

In [5]:

df1 = spark.createDataFrame([
    (101, "Alice", 70000),
    (102, "Bob", 85000),
    (103, "Charlie", 65000),
    (104, "Dan", 90000),
    (105, "Eve", 75000)
], ["emp_id", "name", "salary"])


df1=df1.withColumn('bonus',df1.salary*0.10)
df1.select(df1.emp_id,df1.name,df1.salary,df1.bonus,(df1.salary+df1.bonus).alias('total_salary')).show()

+------+-------+------+------+------------+
|emp_id|   name|salary| bonus|total_salary|
+------+-------+------+------+------------+
|   101|  Alice| 70000|7000.0|     77000.0|
|   102|    Bob| 85000|8500.0|     93500.0|
|   103|Charlie| 65000|6500.0|     71500.0|
|   104|    Dan| 90000|9000.0|     99000.0|
|   105|    Eve| 75000|7500.0|     82500.0|
+------+-------+------+------+------------+



2. Find Top N Salaries per Department

Input: A DataFrame with `emp_id`, `dept`, `salary`
Task: Get top 2 salaries per department using Window functions.

In [6]:
df2 = spark.createDataFrame([
    (101, "HR", 68000),
    (102, "HR", 72000),
    (103, "HR", 60000),
    (201, "IT", 110000),
    (202, "IT", 105000),
    (203, "IT", 95000),
    (301, "Finance", 88000),
    (302, "Finance", 92000)
], ["emp_id", "dept", "salary"])


from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Define window spec partitioned by department and ordered by salary descending
window_spec = Window.partitionBy("dept").orderBy(df2["salary"].desc())

# Add row number to each row per department
ranked_df = df2.withColumn("rank", row_number().over(window_spec))

# Filter top 2 per department
top_n_df = ranked_df.filter(ranked_df["rank"] <= 2)

top_n_df.select("emp_id", "dept", "salary").show()


+------+-------+------+
|emp_id|   dept|salary|
+------+-------+------+
|   302|Finance| 92000|
|   301|Finance| 88000|
|   102|     HR| 72000|
|   101|     HR| 68000|
|   201|     IT|110000|
|   202|     IT|105000|
+------+-------+------+



3. Remove Duplicate Records

Input: A DataFrame with `id`, `name`, `age`, `city`
Task: Remove exact duplicate rows and retain only unique rows.

In [7]:

df3 = spark.createDataFrame([
    (1, "Jane", 28, "Boston"),
    (2, "Mike", 35, "Austin"),
    (3, "Rita", 30, "Denver"),
    (1, "Jane", 28, "Boston"),   # duplicate
    (4, "Sam", 24, "Seattle"),
    (3, "Rita", 30, "Denver")    # duplicate
], ["id", "name", "age", "city"])


df3.dropDuplicates().show()

+---+----+---+-------+
| id|name|age|   city|
+---+----+---+-------+
|  1|Jane| 28| Boston|
|  2|Mike| 35| Austin|
|  3|Rita| 30| Denver|
|  4| Sam| 24|Seattle|
+---+----+---+-------+



4. Get Latest Transaction for Each User

Input: DataFrame with `user_id`, `amount`, `txn_date`
Task: For each user, get their most recent transaction.

In [9]:
df4 = spark.createDataFrame([
    (10, 120.50, "2025-06-01"),
    (10, 215.00, "2025-06-15"),
    (10, 98.75, "2025-06-18"),
    (11, 50.00, "2025-05-30"),
    (11, 75.25, "2025-06-09"),
    (12, 200.00, "2025-06-02")
], ["user_id", "amount", "txn_date"])


from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spc=Window.partitionBy('user_id').orderBy(df4['txn_date'].desc())
ranked_df2=df4.withColumn('rank',row_number().over(window_spc))

top_1_df=ranked_df2.filter(ranked_df2['rank']==1)

top_1_df.select("user_id", "amount", "txn_date").show()

+-------+------+----------+
|user_id|amount|  txn_date|
+-------+------+----------+
|     10| 98.75|2025-06-18|
|     11| 75.25|2025-06-09|
|     12| 200.0|2025-06-02|
+-------+------+----------+



5. Explode Array Column

Input: A DataFrame with columns `user_id`, `hobbies` (ArrayType)
Task: Explode the array to get one row per hobby.

In [12]:
from pyspark.sql.types import ArrayType, StringType

df5 = spark.createDataFrame([
    (1, ["hiking", "photography", "gaming"]),
    (2, ["cooking", "reading"]),
    (3, ["gaming", "running", "chess"])
], ["user_id", "hobbies"])

from pyspark.sql import functions as sf
df5.select(df5['user_id'],sf.explode('hobbies')).show()

+-------+-----------+
|user_id|        col|
+-------+-----------+
|      1|     hiking|
|      1|photography|
|      1|     gaming|
|      2|    cooking|
|      2|    reading|
|      3|     gaming|
|      3|    running|
|      3|      chess|
+-------+-----------+



6. Join Two DataFrames

Input:

- DF1: `emp_id`, `emp_name`
- DF2: `emp_id`, `dept_name`

In [14]:
df6_emp = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David")
], ["emp_id", "emp_name"])

df6_dept = spark.createDataFrame([
    (1, "Finance"),
    (2, "HR"),
    (3, "IT"),
    (5, "Marketing")
], ["emp_id", "dept_name"])

df6_dept.join(df6_dept,'emp_id','inner').show()

+------+---------+---------+
|emp_id|dept_name|dept_name|
+------+---------+---------+
|     1|  Finance|  Finance|
|     2|       HR|       HR|
|     3|       IT|       IT|
|     5|Marketing|Marketing|
+------+---------+---------+



7. GroupBy and Aggregate Multiple Columns

Input: A DataFrame with `region`, `product`, `sales`
Task: Get total sales per region and product.

In [16]:
df7 = spark.createDataFrame([
    ("East", "Gadget", 4500),
    ("East", "Widget", 3000),
    ("West", "Gadget", 5200),
    ("West", "Widget", 2800),
    ("North", "Gadget", 4100),
    ("North", "Widget", 3500)
], ["region", "product", "sales"])

df7.groupBy(df7['region']).count().show()

+------+-----+
|region|count|
+------+-----+
|  East|    2|
|  West|    2|
| North|    2|
+------+-----+



8. Find Users Who Placed Orders More Than 3 Times

Input: DataFrame with `user_id`, `order_id`
Task: Filter users who placed more than 3 orders.

In [17]:
df8 = spark.createDataFrame([
    (101, 5001),
    (101, 5002),
    (101, 5003),
    (101, 5004),
    (102, 6001),
    (102, 6002),
    (103, 7001),
    (103, 7002),
    (103, 7003)
], ["user_id", "order_id"])

df8=df8.groupBy(df8['user_id']).count().alias('count')
df8.filter(df8['count']>3).show()

+-------+-----+
|user_id|count|
+-------+-----+
|    101|    4|
+-------+-----+



9. Null Handling and Fill

Input: A DataFrame with missing values in columns `age`, `salary`
Task: Fill nulls in `age` with 0 and `salary` with the average salary.

In [20]:
df9 = spark.createDataFrame([
    (1, 29, 72000),
    (2, None, 65000),
    (3, 35, None),
    (4, 31, 81000),
    (5, None, None)
], ["id", "age", "salary"])

from pyspark.sql.functions import avg
df9.fillna({'age':0,'salary':df9.select(avg('salary')).collect()[0][0]}).show()

+---+---+------+
| id|age|salary|
+---+---+------+
|  1| 29| 72000|
|  2|  0| 65000|
|  3| 35| 72666|
|  4| 31| 81000|
|  5|  0| 72666|
+---+---+------+



10. Calculate Running Total Using Window Function

Input: DataFrame with `user_id`, `txn_date`, `amount`
Task: For each user, calculate the running total of amount over `txn_date`.

In [2]:
df10 = spark.createDataFrame([
    (10, "2025-06-01", 120),
    (10, "2025-06-05", 180),
    (10, "2025-06-10", 90),
    (11, "2025-06-02", 60),
    (11, "2025-06-12", 75),
    (12, "2025-06-03", 200),
    (12, "2025-06-04", 150)
], ["user_id", "txn_date", "amount"])

from pyspark.sql.window import Window
from pyspark.sql.functions import sum, to_date, col

# Convert txn_date to date type (if needed)
df10 = df10.withColumn("txn_date", to_date("txn_date"))


# Define a window partitioned by user and ordered by date
window_spec = Window.partitionBy("user_id").orderBy("txn_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running total
df_with_running_total = df10.withColumn("running_total", sum("amount").over(window_spec))

df_with_running_total.show()

+-------+----------+------+-------------+
|user_id|  txn_date|amount|running_total|
+-------+----------+------+-------------+
|     10|2025-06-01|   120|          120|
|     10|2025-06-05|   180|          300|
|     10|2025-06-10|    90|          390|
|     11|2025-06-02|    60|           60|
|     11|2025-06-12|    75|          135|
|     12|2025-06-03|   200|          200|
|     12|2025-06-04|   150|          350|
+-------+----------+------+-------------+

